
###### 09_model_serving

###### Purpose

Invoke the deployed Telco Churn model through a Databricks Model Serving endpoint and validate real-time predictions using a sample customer record.

###### Technologies Used

-  Databricks Model Serving

-  MLflow

-  Unity Catalog Model Registry

-  Python Requests

-  REST API

-  Databricks Secrets


###### Input

- Databricks workspace context

- Telco Churn serving endpoint name

- Authentication token from Databricks Secrets

- Sample customer feature payload

######  Output

- HTTP response status

- Real-time churn prediction

- Error details when the request fails


######  Architecture

```text

Sample Customer Features
      ↓
Payload
      ↓
REST API Request
      ↓
Databricks Serving Endpoint
      ↓
Prediction Response

```

###### Section 0 : Load Project Configuration

In [0]:
%run ./00_project_config

###### Section 1 :  Build Serving Endpoint URL

In [0]:
import requests

#Get workspace URL dynamically
workspace_url = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

#print(workspace_url)

#Endpoint configuration
endpoint_url = (
    f"{workspace_url}/serving-endpoints/"
    f"{MODEL_SERVING_ENDPOINT}/invocations"
)


###### Section 2 :  Load Authentication Token

In [0]:
#Authentication credentials are stored in a Databricks secret scope and are never written directly into the notebook.
token = dbutils.secrets.get(
    scope="databricks-secrets",
    key="model-serving-pat"
)

headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}


###### Section 3 :  Prepare Sample Payload

In [0]:
# The serving payload must include Tenure_Group because it was used as a training feature and is therefore part of the model’s expected input schema.

payload = {
    "dataframe_records": [
        {
            "gender": "Female",
            "SeniorCitizen": 0,
            "Partner": "Yes",
            "Dependents": "No",
            "tenure": 24,
            "PhoneService": "Yes",
            "MultipleLines": "No",
            "InternetService": "Fiber optic",
            "OnlineSecurity": "No",
            "OnlineBackup": "No",
            "DeviceProtection": "No",
            "TechSupport": "No",
            "StreamingTV": "Yes",
            "StreamingMovies": "Yes",
            "Contract": "Month-to-month",
            "PaperlessBilling": "Yes",
            "PaymentMethod": "Electronic check",
            "MonthlyCharges": 85.0,
            "TotalCharges": 2100.0,
            "Tenure_Group": "Loyal"
        }
    ]
}


###### Section 4 :  Invoke Serving Endpoint

In [0]:
try:
    response = requests.post(
        endpoint_url,
        headers=headers,
        json=payload,
        timeout=60
    )

    response.raise_for_status()
    prediction_response = response.json()

    print("Serving endpoint invoked successfully.")

except requests.exceptions.Timeout as exc:
    raise RuntimeError(
        "The model serving request timed out."
    ) from exc

except requests.exceptions.RequestException as exc:
    raise RuntimeError(
        f"Model serving request failed: {exc}"
    ) from exc

predictions = prediction_response.get(
    "predictions",
    None
)

###### Section 5: Validate Prediction Response

In [0]:
if not predictions:
    raise ValueError(
        f"No predictions were returned: {prediction_response}"
    )

print("Serving request completed successfully.")

print(f"Predicted churn class: {predictions[0]}")


##### Notebook Summary

-  Retrieved the Databricks workspace URL dynamically.

-  Constructed the Model Serving invocation URL.

-  Loaded authentication securely from Databricks Secrets.

-  Created a sample customer-feature payload that matches the model signature.

-  Submitted a real-time inference request through the REST API.

-  Validated the prediction response and handled request errors.


######  Key Learnings

-  A serving payload must match the model signature created during training.

-  Feature-engineered columns such as Tenure_Group must be supplied when they are part of the model input schema.

-  Databricks Model Serving exposes registered models through a REST API for real-time inference.

-  Authentication values should be stored in Databricks Secrets rather than hard-coded in notebooks.

-  HTTP status validation and exception handling are required for reliable endpoint integration.

###### Notebook Conclusion

- In this notebook, we invoked the deployed Telco Churn model through a Databricks Model Serving endpoint and validated a real-time churn prediction using a sample customer record.

- The successful REST API response confirms that the registered model can be consumed by downstream applications and AI agents.

###### Next Notebook

10_embeddings

- Generate dense vector representations of customer notes using an embedding model. These embeddings will support semantic retrieval, Vector Search, RAG, and later Agentic AI workflows.